In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

In [2]:
df = pd.read_csv('../data/processed/ratings_clean.csv')
movies = pd.read_csv('../data/raw/movies.csv')
tags = pd.read_csv('../data/raw/tags.csv')

print(df.shape, movies.shape, tags.shape)

(100836, 6) (9742, 3) (3683, 4)


In [3]:
# collaborative filtering
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(df[['userId', 'movieId', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
svd = SVD(n_factors=100, random_state=42)
svd.fit(trainset)

# content-based
tags_clean = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()
tags_clean.columns = ['movieId', 'tags']
movies = movies.merge(tags_clean, on='movieId', how='left')
movies['tags'] = movies['tags'].fillna('')
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)
movies['features'] = movies['genres_clean'] + ' ' + movies['tags']

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['features'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Both models ready!")

Both models ready!


In [6]:
def hybrid_recommendations(user_id, n=10, alpha=0.7):
    rated_movies = df[df['userId'] == user_id]['movieId'].tolist()
    all_movies = movies['movieId'].unique()
    unrated = [m for m in all_movies if m not in rated_movies]

    results = []
    for movie_id in unrated:
        cf_score = svd.predict(user_id, movie_id).est

        top5 = (df[df['userId'] == user_id]
                .sort_values('rating', ascending=False)
                .head(5)['movieId'].tolist())
        
        idx = movies[movies['movieId'] == movie_id].index
        if len(idx) == 0:
            continue
        idx = idx[0]
        
        cb_scores = []
        for rated_id in top5:
            rated_idx = movies[movies['movieId'] == rated_id].index
            if len(rated_idx) > 0:
                cb_scores.append(cosine_sim[idx][rated_idx[0]])
        
        cb_score = np.mean(cb_scores) if cb_scores else 0
        hybrid_score = alpha * cf_score + (1 - alpha) * cb_score
        results.append((movie_id, hybrid_score))

    results.sort(key=lambda x: x[1], reverse=True)
    top_n = results[:n]

    movie_ids = [r[0] for r in top_n]
    scores = [round(r[1], 3) for r in top_n]

    result = movies[movies['movieId'].isin(movie_ids)].copy()
    result['score'] = result['movieId'].map(dict(zip(movie_ids, scores)))
    result = result.sort_values('score', ascending=False).reset_index(drop=True)
    result.insert(0, 'rank', range(1, len(result) + 1))
    return result[['rank', 'title', 'genres', 'score']]

In [7]:
hybrid_recommendations(user_id=1)

,rank,title,genres,score
0,1,Cinema Paradiso (Nuovo cinema Paradiso) (1989),Drama,3.545
1,2,Seven Samurai (Shichinin no samurai) (1954),Action|Adventure|Drama,3.540
2,3,"Boot, Das (Boat, The) (1981)",Action|Drama|War,3.539
3,4,Lawrence of Arabia (1962),Adventure|Drama|War,3.533
4,5,"Boondock Saints, The (2000)",Action|Crime|Drama|Thriller,3.532
5,6,Dancer in the Dark (2000),Drama|Musical,3.524
6,7,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,3.523
7,8,"Grand Day Out with Wallace and Gromit, A (1989)",Adventure|Animation|Children|Comedy|Sci-Fi,3.519
8,9,North by Northwest (1959),Action|Adventure|Mystery|Romance|Thriller,3.517
9,10,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,3.513
